![cyber_photo](cyber_photo.jpg)

Cyber threats are a growing concern for organizations worldwide. These threats take many forms, including malware, phishing, and denial-of-service (DOS) attacks, compromising sensitive information and disrupting operations. The increasing sophistication and frequency of these attacks make it imperative for organizations to adopt advanced security measures. Traditional threat detection methods often fall short due to their inability to adapt to new and evolving threats. This is where deep learning models come into play.

Deep learning models can analyze vast amounts of data and identify patterns that may not be immediately obvious to human analysts. By leveraging these models, organizations can proactively detect and mitigate cyber threats, safeguarding their sensitive information and ensuring operational continuity.

As a cybersecurity analyst, you identify and mitigate these threats. In this project, you will design and implement a deep learning model to detect cyber threats. The BETH dataset simulates real-world logs, providing a rich source of information for training and testing your model. The data has already undergone preprocessing, and we have a target label, `sus_label`, indicating whether an event is malicious (1) or benign (0).

By successfully developing this model, you will contribute to enhancing cybersecurity measures and protecting organizations from potentially devastating cyber attacks.


### The Data

| Column     | Description              |
|------------|--------------------------|
|`processId`|The unique identifier for the process that generated the event - int64 |
|`threadId`|ID for the thread spawning the log - int64|
|`parentProcessId`|Label for the process spawning this log - int64|
|`userId`|ID of user spawning the log|Numerical - int64|
|`mountNamespace`|Mounting restrictions the process log works within - int64|
|`argsNum`|Number of arguments passed to the event - int64|
|`returnValue`|Value returned from the event log (usually 0) - int64|
|`sus_label`|Binary label as suspicous event (1 is suspicious, 0 is not) - int64|

More information on the dataset: [BETH dataset](accreditation.md)

In [163]:
# Import required libraries
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as functional
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from torchmetrics import Accuracy
# from sklearn.metrics import accuracy_score  # uncomment to use sklearn

In [164]:
# Load preprocessed data
train_df = pd.read_csv('labelled_train.csv')
test_df = pd.read_csv('labelled_test.csv')
val_df = pd.read_csv('labelled_validation.csv')

# View the first 5 rows of training set
train_df.head()

,processId,threadId,parentProcessId,userId,mountNamespace,argsNum,returnValue,sus_label
0,381,7337,1,100,4026532231,5,0,1
1,381,7337,1,100,4026532231,1,0,1
2,381,7337,1,100,4026532231,0,0,1
3,7347,7347,7341,0,4026531840,2,-2,1
4,7347,7347,7341,0,4026531840,4,0,1


In [165]:
# Start coding here
# Use as many cells as you need

In [166]:
#separating the data

X=train_df.drop("sus_label",axis=1).values
Y=train_df["sus_label"].values

In [167]:
#scaling the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X)


In [168]:
# convert to pytorch
X_train=torch.tensor(X_train).float()
Y_train=torch.tensor(Y).float()
print(Y_train.shape)
Y_train = Y_train.reshape(-1,1)
print(train_df["sus_label"].value_counts(normalize=True))
print(test_df["sus_label"].value_counts(normalize=True))
print(val_df["sus_label"].value_counts(normalize=True))

torch.Size([763144])
0    0.998337
1    0.001663
Name: sus_label, dtype: float64
1    0.907349
0    0.092651
Name: sus_label, dtype: float64
0    0.995841
1    0.004159
Name: sus_label, dtype: float64


In [169]:
#create the model

model= nn.Sequential(nn.Linear(7,128),
                    nn.ReLU(),
                   nn.Linear(128,64),
                     nn.ReLU(),
                    nn.Linear(64,1),
                   nn.Sigmoid())
#loss function and optimizer
criterion=nn.BCELoss()
optimizer=optim.SGD(model.parameters(),lr=1e-3,weight_decay=1e-4)

num_epoch = 10
for epoch in range(num_epoch):
    model.train()
    optimizer.zero_grad()  
    pred = model(X_train)
    loss = criterion(pred, Y_train)
    loss.backward()
    optimizer.step()

In [170]:
#evaluating the modal
model.eval()
metrica =Accuracy(task="binary")

#on the test set

X=test_df.drop("sus_label",axis=1).values
Y=test_df["sus_label"].values
X=scaler.transform(X)
X_test=torch.tensor(X).float()
Y_test=torch.tensor(Y).float()
Y_test = Y_test.reshape(-1,1)
#on the validatition set
X=val_df.drop("sus_label",axis=1).values
Y=val_df["sus_label"].values
X=scaler.transform(X)
X_val=torch.tensor(X).float()
Y_val=torch.tensor(Y).float()
Y_val = Y_val.reshape(-1,1)

eval_model={"train":[X_train,Y_train], "test": [X_test,Y_test], "evaluation":[X_val,Y_val]}
evalu=dict()
for dataset in eval_model:
        metrica.reset() 
        X=eval_model[dataset][0]
        Y=eval_model[dataset][1]
        with torch.no_grad():
            pred=model(X)
            accu=metrica(pred,Y)
            evalu[dataset]=accu
print(evalu)
val_accuracy=float(evalu["evaluation"])
print(val_accuracy)


{'train': tensor(0.9983), 'test': tensor(0.0927), 'evaluation': tensor(0.9958)}
0.9958405494689941
